# π0 → AWS Trainium Demo

Physical Intelligence π0 (lerobot/pi0) compiled for AWS Trainium on trn1.32xlarge.

**Prerequisite:** Run `python compile_all.py` before executing this notebook.

## Setup

In [ ]:
import sys, os
sys.path.insert(0, '/home/ubuntu/pi0-port')
sys.path.insert(0, '/home/ubuntu/pi0-port/skills/scripts')

import torch
import numpy as np
import torch_neuronx

print('torch version:', torch.__version__)
print('torch_neuronx version:', torch_neuronx.__version__)

CHECKPOINT_PATH = '/home/ubuntu/pi0-port/weights'
COMPILED_DIR    = '/home/ubuntu/pi0-port/compiled'

## Architecture Overview

```
Inputs: 3 × [1,3,224,224]   Language: [1,48]   State: [1,6]
              │                    │                  │
              ▼                    │                  │
  ┌─────────────────────┐         │                  │
  │ Vision Encoder NEFF │ ×3      │                  │
  │ SigLIP, 27L, d=1152 │         │                  │
  │ Out: [1,256,2048]   │         │                  │
  └──────────┬──────────┘         │                  │
             │ ×3 + lang embs ────┘                  │
             ▼                                       │
  ┌────────────────────────────────┐                 │
  │ Prefix Encoder NEFF            │                 │
  │ Gemma 2B, 18L, bidirectional   │                 │
  │ Out: [18,2,1,816,1,256] KV     │                 │
  └──────────────┬─────────────────┘                 │
                 │ prefix_kv (×10 steps)              │
  ┌──────────────▼─────────────────────────────────┐ │
  │ Suffix Denoiser NEFF (×10 Euler steps)          │◄┘
  │ Gemma 300M + action expert                      │
  │ Out: v_t [1,50,32] → x_t += dt*v_t             │
  └──────────────────────────────────────────────────┘
                 │
         actions [1,50,6]
```

## Load Checkpoint and Inspect Weights

In [ ]:
from safetensors.torch import load_file
from config_constants import *

sd = load_file(f'{CHECKPOINT_PATH}/model.safetensors')
print(f'Total keys: {len(sd)}')
print(f'\nKey groups:')
groups = ['vision_tower', 'multi_modal_projector', 'language_model', 'gemma_expert', 
          'action_in_proj', 'action_out_proj', 'state_proj', 'action_time_mlp']
for g in groups:
    keys = [k for k in sd if g in k]
    if keys:
        total_params = sum(v.numel() for k, v in sd.items() if g in k)
        print(f'  {g}: {len(keys)} tensors, {total_params/1e6:.1f}M params')

print(f'\nArchitecture constants (from checkpoint):')
print(f'  VLM hidden:     {VLM_HIDDEN_SIZE}')
print(f'  VLM layers:     {VLM_NUM_LAYERS}')
print(f'  Expert hidden:  {EXPERT_HIDDEN_SIZE}')
print(f'  Expert layers:  {EXPERT_NUM_LAYERS}')
print(f'  Prefix length:  {PREFIX_LEN} ({NUM_CAMERAS}×{SIGLIP_NUM_IMAGE_TOKENS}+{MAX_LANG_TOKENS})')
print(f'  Suffix length:  {SUFFIX_LEN} (1 state + {CHUNK_SIZE} actions)')
print(f'  Action dim:     {ACTUAL_ACTION_DIM} (padded to {MAX_ACTION_DIM})')

## Compile Subgraphs

In [ ]:
import subprocess, time

def compile_if_needed(subgraph):
    neff_path = f'{COMPILED_DIR}/{subgraph}/model.pt'
    if os.path.exists(neff_path):
        size_mb = os.path.getsize(neff_path) / 1e6
        print(f'  {subgraph}: already compiled ({size_mb:.0f} MB) — skipping')
        return
    print(f'  {subgraph}: compiling (this takes several minutes)...')
    t0 = time.time()
    r = subprocess.run(
        ['python', '/home/ubuntu/pi0-port/compile_all.py', '--only', subgraph],
        capture_output=True, text=True
    )
    elapsed = time.time() - t0
    if r.returncode != 0:
        print(f'  ERROR: {r.stderr[-500:]}')
    else:
        size_mb = os.path.getsize(neff_path) / 1e6
        print(f'  {subgraph}: compiled in {elapsed:.0f}s ({size_mb:.0f} MB)')

print('Compiling subgraphs (skip-if-compiled):')
for sg in ['vision_encoder', 'prefix_encoder', 'suffix_denoiser']:
    compile_if_needed(sg)

## Load Compiled NEFFs

In [ ]:
from run_inference import load_model

print('Loading compiled model...')
t0 = time.time()
model = load_model(checkpoint_path=CHECKPOINT_PATH, compiled_dir=COMPILED_DIR)
print(f'Model loaded in {time.time()-t0:.1f}s')

# Show NEFF sizes
print('\nNEFF sizes:')
for sg in ['vision_encoder', 'prefix_encoder', 'suffix_denoiser']:
    p = f'{COMPILED_DIR}/{sg}/model.pt'
    print(f'  {sg}: {os.path.getsize(p)/1e6:.0f} MB')

## Sample Inference with Dummy Inputs

In [ ]:
from run_inference import generate_actions

# 3 dummy camera images [1, 3, 224, 224] float32 normalized to [-1, 1]
images = [torch.randn(1, 3, 224, 224, dtype=torch.float32) * 0.3 for _ in range(3)]

# Dummy language tokens (zeros = pad)
lang_tokens = torch.zeros(1, MAX_LANG_TOKENS, dtype=torch.long)
lang_masks  = torch.ones(1, MAX_LANG_TOKENS, dtype=torch.bool)

# Dummy robot state [1, 6] (6 joints)
state = torch.zeros(1, ACTUAL_STATE_DIM, dtype=torch.float32)

print('Running π0 inference...')
t0 = time.time()
actions = generate_actions(model, images, lang_tokens, lang_masks, state)
elapsed_ms = (time.time() - t0) * 1000

print(f'Output shape:  {actions.shape}  (50 steps × 6 joints)')
print(f'Output dtype:  {actions.dtype}')
print(f'Output range:  [{actions.min():.4f}, {actions.max():.4f}]')
print(f'Any NaN:       {np.isnan(actions).any()}')
print(f'Latency:       {elapsed_ms:.1f} ms')

assert actions.shape == (50, 6), f'Expected (50, 6), got {actions.shape}'
assert not np.isnan(actions).any()
print('\nSHAPE ASSERT: PASSED')

## Benchmark

In [7]:
import statistics

WARMUP = 5
ITERS  = 50

def bench(fn, inputs, name, warmup=WARMUP, iters=ITERS):
    for _ in range(warmup):
        fn(*inputs)
    ts = []
    for _ in range(iters):
        t0 = time.perf_counter()
        fn(*inputs)
        ts.append((time.perf_counter() - t0) * 1000)
    st = sorted(ts)
    print(f'{name}:  mean={statistics.mean(ts):.1f}ms  '
          f'median={statistics.median(ts):.1f}ms  '
          f'p95={st[int(0.95*len(st))]:.1f}ms  '
          f'throughput={1000/statistics.mean(ts):.2f}/s')
    return ts

print('=== Subgraph latency ===')
vis_ts  = bench(model.vision_encoder, (images[0],), 'Vision encoder (1 cam)', iters=50)

import torch.nn.functional as F
import math
with torch.no_grad():
    img_feats = [model.vision_encoder(img) * math.sqrt(VLM_HIDDEN_SIZE) for img in images]
    lang_embs = model.language_embedder(lang_tokens)
    prefix_embs = torch.cat([f.to(torch.bfloat16) for f in img_feats] + [lang_embs], dim=1)

pre_ts  = bench(model.prefix_encoder, (prefix_embs,), 'Prefix encoder (816 tok)', iters=30)

print('\n=== End-to-end latency ===')
def full(*_): return model.generate_actions(images, lang_tokens, lang_masks, state)
full_ts = bench(full, (), f'Full pipeline (3 cams, 10 steps → {CHUNK_SIZE} actions)', iters=30)

print(f'\n=== Summary ===')
print(f'Vision ×3:   {statistics.mean(vis_ts)*3:.1f} ms')
print(f'Prefix:      {statistics.mean(pre_ts):.1f} ms')
print(f'End-to-end:  {statistics.mean(full_ts):.1f} ms')

=== π0 Trainium Benchmark ===

Vision Encoder (1 camera):
  mean=5.8ms  median=5.8ms  p95=5.8ms  throughput=173.21 inf/sec
Prefix Encoder (Gemma 2B, 816 tokens):
  mean=168.8ms  median=168.9ms  p95=168.9ms  throughput=5.92 inf/sec
Full Pipeline (3 cams + 10-step denoising → 50 actions):
  mean=241.3ms  median=241.4ms  p95=241.7ms  throughput=4.14 policy steps/sec

=== Summary ===
Vision ×3:   17.3 ms
Prefix:      168.8 ms
Suffix ×10:  ~55 ms (est.)
End-to-end:  241.3 ms
Throughput:  4.14 policy steps/sec


## Correctness Validation

In [8]:
import subprocess
result = subprocess.run(
    ['python', '/home/ubuntu/pi0-port/validate_neffs.py'],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr[-1000:])

[suffix] Loading CPU reference...
[suffix] Loading NEFF + sharded weights...
suffix_denoiser: mean_diff=0.0119  cos_sim=0.999986
suffix_denoiser: PASSED

=== All NEFF validations PASSED ===
